# Dataset selection and ECG inspection

This combines the useful dataset-selection work from `overview`, `explore`, `fors`, and `incent`. PhysioNet 2017 supplies the main binary task; MIMIC-IV and Icentia11k remain optional investigations.

Install the project first with `pip install -e ".[neural,signal,wavelet,notebooks]"`. Opening this notebook does not start training or download data.


In [ ]:
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
print(f"Project: {ROOT.name}")


## Main dataset

The project reader keeps normal and AFib records. The other-rhythm and noisy classes are outside the current binary experiments.


In [ ]:
from ecg_afib.datasets import summarize_physionet

raw = ROOT / "data/raw/training2017"
if (raw / "REFERENCE.csv").exists():
    print(summarize_physionet(raw))
else:
    print("Place REFERENCE.csv and the named .mat files in data/raw/training2017/.")


## Inspect a recording and its spectrum

This is descriptive signal exploration; it does not fit a model or select a test-set threshold.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ecg_afib.data import read_references, read_signal

if (raw / "REFERENCE.csv").exists():
    record = read_references(raw)[0]
    signal = read_signal(raw / f"{record['record_id']}.mat")[:9000]
    fs = 300
    fig, axes = plt.subplots(2, 1, figsize=(10, 5))
    axes[0].plot(np.arange(len(signal)) / fs, signal)
    axes[0].set(xlabel="Time (s)", ylabel="ECG amplitude", title=record["record_id"])
    axes[1].plot(np.fft.rfftfreq(len(signal), 1 / fs), np.abs(np.fft.rfft(signal)) / len(signal))
    axes[1].set(xlim=(0, 40), xlabel="Frequency (Hz)", ylabel="Magnitude")
    fig.tight_layout()
else:
    print("Signal plots become available after adding the dataset.")


## Optional WFDB sources

Set explicit record locations only if you want to revisit these sources. The helpers read actual record metadata instead of assuming a sampling frequency. No outputs from this section feed the main experiment presets.


In [ ]:
from ecg_afib.datasets import inspect_wfdb, inspect_annotations

MIMIC_LOCAL_RECORD = None  # Path to a local WFDB record, without extension.
ICENTIA_RECORD = None      # A record name chosen from the official dataset.
ICENTIA_DIRECTORY = None   # Its corresponding PhysioNet directory.

if MIMIC_LOCAL_RECORD is not None:
    print(inspect_wfdb(MIMIC_LOCAL_RECORD))
if ICENTIA_RECORD is not None and ICENTIA_DIRECTORY is not None:
    print(inspect_wfdb(ICENTIA_RECORD, pn_dir=ICENTIA_DIRECTORY))
    print(inspect_annotations(ICENTIA_RECORD, pn_dir=ICENTIA_DIRECTORY))


The incomplete Icentia interval exporter and local pickle probes were removed during consolidation. See [the migration map](../docs/MIGRATION.md) for the exact decisions.
